<a href="https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/02-python-para-analistas-soc/02_python_para_analistas_soc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 02 — Python para Analistas de SOC

**Escenario:** Tenemos un dump de logs de autenticación SSH de las últimas 6 horas.

Tu trabajo: procesarlos con Python, identificar IPs sospechosas, detectar patrones de fuerza bruta, y generar un resumen para el siguiente turno.

Objetivos:
- Leer y procesar logs con Python
- Extraer campos con regex
- Detectar fuerza bruta con lógica simple
- Encapsular lógica en funciones reutilizables
- Aplicar buenas prácticas: manejo de errores, logging, credenciales
- **Modularidad**: organizar el código en módulos reutilizables por responsabilidad
- **Resiliencia**: manejar fallos de dependencias externas con reintentos
- **Testing con pytest**: verificar que el detector funciona antes de ponerlo en producción

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/python_for_logs.png" width="460"/>

In [1]:
import re
import json
import logging
import os
from datetime import datetime
from collections import Counter, defaultdict
from pathlib import Path

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger('soc-demo')

print('Imports OK')

Imports OK


In [2]:
import sys
import os
from pathlib import Path

# ─── Detección de entorno: Colab vs local ────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # En Colab: clonar el repo completo para tener el paquete soc/ disponible
    REPO_URL = 'https://github.com/florvela/IA-y-automatizacion-en-seguridad-defensiva'
    REPO_DIR = '/content/IA-y-automatizacion-en-seguridad-defensiva'
    MODULE_PATH = f'{REPO_DIR}/codigos-de-ejemplo/02-python-para-analistas-soc'
    if not Path(REPO_DIR).exists():
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)
    print(f'Colab: clonado {REPO_URL}')
    print(f'       sys.path apunta a {MODULE_PATH}')
else:
    # Local: el notebook ya está en la carpeta del módulo
    MODULE_PATH = str(Path('.').resolve())
    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)
    print(f'Local: sys.path apunta a {MODULE_PATH}')

print(f'Entorno: {"Google Colab" if IN_COLAB else "local"} — listo')

Colab: clonado https://github.com/florvela/IA-y-automatizacion-en-seguridad-defensiva
       sys.path apunta a /content/IA-y-automatizacion-en-seguridad-defensiva/codigos-de-ejemplo/02-python-para-analistas-soc
Entorno: Google Colab — listo


## 1. Datos de muestra: logs SSH simulados

En producción estos logs vienen de `/var/log/auth.log` o de tu SIEM. Para la demo usamos una muestra representativa que incluye tipos de eventos que vas a ver en un SOC.

In [3]:
LOGS_MUESTRA = """
Dec 10 06:55:46 LabSZ sshd[24200]: Invalid user webmaster from 173.234.31.186
Dec 10 06:55:48 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2
Dec 10 06:55:48 LabSZ sshd[24200]: Connection closed by 173.234.31.186 [preauth]
Dec 10 07:02:47 LabSZ sshd[24203]: Connection closed by 212.47.254.145 [preauth]
Dec 10 07:07:38 LabSZ sshd[24206]: Invalid user test9 from 52.80.34.196
Dec 10 07:07:40 LabSZ sshd[24206]: Failed password for invalid user test9 from 52.80.34.196 port 45123 ssh2
Dec 10 07:12:14 LabSZ sshd[24210]: Accepted password for admin from 192.168.1.50 port 52341 ssh2
Dec 10 07:15:22 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39001 ssh2
Dec 10 07:15:25 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39002 ssh2
Dec 10 07:15:28 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39003 ssh2
Dec 10 07:15:31 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39004 ssh2
Dec 10 07:15:34 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39005 ssh2
Dec 10 07:15:37 LabSZ sshd[24211]: Failed password for root from 173.234.31.186 port 39006 ssh2
Dec 10 07:18:00 LabSZ sshd[24215]: reverse mapping checking getaddrinfo for ns.marryaldkfaczcz.com [173.234.31.186] failed - POSSIBLE BREAK-IN ATTEMPT!
Dec 10 07:20:11 LabSZ sshd[24220]: Accepted publickey for deploy from 10.0.0.15 port 60001 ssh2
Dec 10 07:22:45 LabSZ sshd[24225]: Invalid user oracle from 45.33.32.156
Dec 10 07:22:47 LabSZ sshd[24225]: Failed password for invalid user oracle from 45.33.32.156 port 55000 ssh2
Dec 10 07:25:03 LabSZ sshd[24230]: Invalid user postgres from 45.33.32.156
Dec 10 07:25:05 LabSZ sshd[24230]: Failed password for invalid user postgres from 45.33.32.156 port 55100 ssh2
Dec 10 07:30:00 LabSZ sshd[24235]: session opened for user admin by (uid=0)
""".strip().split('\n')

print(f'Total de líneas de log: {len(LOGS_MUESTRA)}')
print('\nPrimeras 3 líneas:')
for linea in LOGS_MUESTRA[:3]:
    print(f'  {linea}')

Total de líneas de log: 20

Primeras 3 líneas:
  Dec 10 06:55:46 LabSZ sshd[24200]: Invalid user webmaster from 173.234.31.186
  Dec 10 06:55:48 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2
  Dec 10 06:55:48 LabSZ sshd[24200]: Connection closed by 173.234.31.186 [preauth]


## 2. Regex para extraer campos de logs SSH

Cada línea de log SSH tiene una estructura predecible. Con regex podemos extraer cada campo en forma confiable.

Formato: `MES DIA HH:MM:SS HOST PROCESO[PID]: MENSAJE`

## Regex SSH (failed password)

```regex
(?P<mes>\w+)\s+
(?P<dia>\d+)\s+
(?P<hora>[\d:]+)\s+
\S+\s+
\S+:\s+
Failed password for (?:invalid user )?
(?P<usuario>\S+)
from (?P<ip>[\d.]+)
port (?P<puerto>\d+)
````

### Parte 1 → Header del log

```regex
(?P<mes>\w+)\s+
(?P<dia>\d+)\s+
(?P<hora>[\d:]+)\s+
\S+\s+
\S+:\s+
```

Extrae: mes, día, hora, hostname y proceso.

### Parte 2 → Evento SSH

```regex
Failed password for (?:invalid user )?(?P<usuario>\S+) from (?P<ip>[\d.]+) port (?P<puerto>\d+)
```

* `"Failed password for"` → texto literal
* `(?:invalid user )?` → opcional
* `(?P<usuario>\S+)` → usuario
* `(?P<ip>[\d.]+)` → IP
* `(?P<puerto>\d+)` → puerto

🔧 Testear: [https://regex101.com/](https://regex101.com/)

In [4]:
# Patrones regex para los distintos tipos de eventos SSH
PATRONES = {
    'failed_password': re.compile(
        r'(?P<mes>\w+)\s+(?P<dia>\d+)\s+(?P<hora>[\d:]+)\s+\S+\s+\S+:\s+'
        r'Failed password for (?:invalid user )?(?P<usuario>\S+) from (?P<ip>[\d.]+) port (?P<puerto>\d+)'
    ),
    'invalid_user': re.compile(
        r'(?P<mes>\w+)\s+(?P<dia>\d+)\s+(?P<hora>[\d:]+)\s+\S+\s+\S+:\s+'
        r'Invalid user (?P<usuario>\S+) from (?P<ip>[\d.]+)'
    ),
    'accepted': re.compile(
        r'(?P<mes>\w+)\s+(?P<dia>\d+)\s+(?P<hora>[\d:]+)\s+\S+\s+\S+:\s+'
        r'Accepted (?P<metodo>\S+) for (?P<usuario>\S+) from (?P<ip>[\d.]+) port (?P<puerto>\d+)'
    ),
    'break_in': re.compile(
        r'(?P<mes>\w+)\s+(?P<dia>\d+)\s+(?P<hora>[\d:]+).*'
        r'POSSIBLE BREAK-IN ATTEMPT.*\[(?P<ip>[\d.]+)\]'
    ),
}

def parsear_log_ssh(linea: str) -> dict | None:
    """Parsea una línea de log SSH y devuelve un dict con los campos extraídos."""
    for tipo, patron in PATRONES.items():
        match = patron.search(linea)
        if match:
            datos = match.groupdict()
            datos['tipo'] = tipo
            datos['raw'] = linea
            return datos
    return None

# Demo: parsear las primeras líneas
print('Parseando logs...\n')
for linea in LOGS_MUESTRA[:5]:
    resultado = parsear_log_ssh(linea)
    if resultado:
        print(f"Tipo: {resultado['tipo']}")
        print(f"  IP: {resultado.get('ip', 'N/A')} | Usuario: {resultado.get('usuario', 'N/A')}")
    else:
        print(f'Sin parsear: {linea[:60]}...')
    print()

Parseando logs...

Tipo: invalid_user
  IP: 173.234.31.186 | Usuario: webmaster

Tipo: failed_password
  IP: 173.234.31.186 | Usuario: webmaster

Sin parsear: Dec 10 06:55:48 LabSZ sshd[24200]: Connection closed by 173....

Sin parsear: Dec 10 07:02:47 LabSZ sshd[24203]: Connection closed by 212....

Tipo: invalid_user
  IP: 52.80.34.196 | Usuario: test9



## 3. Análisis con estructuras de datos

`Counter`

In [5]:
# Parsear todos los logs
eventos = [parsear_log_ssh(l) for l in LOGS_MUESTRA]
eventos = [e for e in eventos if e is not None]

print(f'Eventos parseados: {len(eventos)} de {len(LOGS_MUESTRA)} líneas')

# Conteo por tipo de evento
tipos = Counter(e['tipo'] for e in eventos)
print('\n--- Distribución por tipo ---')
for tipo, count in tipos.most_common():
    print(f'  {tipo:<25} {count:>3} eventos')

# IPs con más intentos fallidos
intentos_por_ip = Counter(
    e['ip'] for e in eventos
    if e['tipo'] in ('failed_password', 'invalid_user', 'break_in')
)
print('\n--- Top IPs sospechosas ---')
for ip, count in intentos_por_ip.most_common(5):
    print(f'  {ip:<20} {count:>3} intentos fallidos')

# Usuarios más atacados
usuarios_atacados = Counter(
    e.get('usuario', 'unknown') for e in eventos
    if e['tipo'] == 'failed_password'
)
print('\n--- Usuarios más atacados ---')
for usuario, count in usuarios_atacados.most_common(5):
    print(f'  {usuario:<20} {count:>3} intentos')

Eventos parseados: 16 de 20 líneas

--- Distribución por tipo ---
  failed_password            10 eventos
  invalid_user                4 eventos
  accepted                    2 eventos

--- Top IPs sospechosas ---
  173.234.31.186         8 intentos fallidos
  45.33.32.156           4 intentos fallidos
  52.80.34.196           2 intentos fallidos

--- Usuarios más atacados ---
  root                   6 intentos
  webmaster              1 intentos
  test9                  1 intentos
  oracle                 1 intentos
  postgres               1 intentos


## 4. Detección de fuerza bruta

Un patrón que buscamos: la misma IP genera más de N intentos fallidos en un período corto.

In [6]:
from collections import defaultdict

def detectar_fuerza_bruta(eventos: list, umbral: int = 5) -> list:
    """
    Detecta IPs con comportamiento de fuerza bruta.
    Retorna lista de dicts con IP, cantidad de intentos y usuarios probados.
    """
    intentos_por_ip = defaultdict(dict)

    for e in eventos:
        if e['tipo'] == 'failed_password' or e['tipo'] == 'invalid_user':
            ip = e.get('ip', 'unknown')

            if ip not in intentos_por_ip:
                intentos_por_ip[ip] = {
                    'count': 0,
                    'usuarios': set(),
                    'eventos': []
                }

            intentos_por_ip[ip]['count'] += 1
            intentos_por_ip[ip]['usuarios'].add(e.get('usuario', 'unknown'))
            intentos_por_ip[ip]['eventos'].append(e)

    sospechosas = []

    for ip, datos in intentos_por_ip.items():
        if datos['count'] >= umbral:

            if datos['count'] >= umbral * 2:
                riesgo = 'ALTO'
            else:
                riesgo = 'MEDIO'

            sospechosas.append({
                'ip': ip,
                'intentos': datos['count'],
                'usuarios_probados': list(datos['usuarios']),
                'riesgo': riesgo
            })

    # Ordenar manualmente (sin lambda)
    def obtener_intentos(item):
        return item['intentos']

    sospechosas_ordenadas = sorted(sospechosas, key=obtener_intentos, reverse=True)

    return sospechosas_ordenadas


def generar_resumen(eventos: list) -> dict:
    """Genera un resumen ejecutivo del análisis de logs."""
    total = len(eventos)

    fallidos = 0
    exitosos = 0

    for e in eventos:
        if e['tipo'] == 'failed_password':
            fallidos += 1
        if e['tipo'] == 'accepted':
            exitosos += 1

    sospechosas = detectar_fuerza_bruta(eventos)

    if total > 0:
        tasa_exito = (exitosos / total) * 100
        tasa_exito_str = f'{tasa_exito:.1f}%'
    else:
        tasa_exito_str = '0%'

    return {
        'total_eventos': total,
        'intentos_fallidos': fallidos,
        'accesos_exitosos': exitosos,
        'tasa_exito': tasa_exito_str,
        'ips_fuerza_bruta': len(sospechosas),
        'detalle_sospechosas': sospechosas
    }


# Ejecutar detección
sospechosas = detectar_fuerza_bruta(eventos, umbral=3)

print(f'IPs con comportamiento de fuerza bruta (umbral=3): {len(sospechosas)}')
print()

for ip_info in sospechosas:
    print(f"  IP: {ip_info['ip']}")
    print(f"    Intentos: {ip_info['intentos']} | Riesgo: {ip_info['riesgo']}")
    print(f"    Usuarios probados: {ip_info['usuarios_probados']}")
    print()

IPs con comportamiento de fuerza bruta (umbral=3): 2

  IP: 173.234.31.186
    Intentos: 8 | Riesgo: ALTO
    Usuarios probados: ['webmaster', 'root']

  IP: 45.33.32.156
    Intentos: 4 | Riesgo: MEDIO
    Usuarios probados: ['oracle', 'postgres']



## 5. Leyendo desde archivo con manejo de errores

En producción los logs vienen de archivos o streams.

Los podemos leer correctamente con manejo de errores, encoding explícito y `pathlib`.

In [7]:
def leer_logs_desde_archivo(ruta: str | Path) -> list[str]:
    """
    Lee un archivo de logs línea por línea.
    Maneja errores de encoding, archivo no encontrado, y permisos.
    """
    ruta = Path(ruta)

    if not ruta.exists():
        logger.warning(f'Archivo no encontrado: {ruta}')
        return []

    if not ruta.is_file():
        logger.error(f'La ruta no es un archivo: {ruta}')
        return []

    lineas = []
    try:
        with open(ruta, 'r', encoding='utf-8', errors='replace') as f:
            for numero, linea in enumerate(f, start=1):
                linea = linea.strip()
                if linea:  # Ignorar líneas vacías
                    lineas.append(linea)
        logger.info(f'Leídas {len(lineas)} líneas desde {ruta.name}')
    except PermissionError:
        logger.error(f'Sin permiso para leer: {ruta}')
    except Exception as e:
        logger.error(f'Error inesperado leyendo {ruta}: {e}')

    return lineas


# Demo: guardar la muestra en un archivo y releerla
archivo_temp = Path('demo_auth.log')
with open(archivo_temp, 'w') as f:
    f.write('\n'.join(LOGS_MUESTRA))

logs_desde_archivo = leer_logs_desde_archivo(archivo_temp)
print(f'Leídas {len(logs_desde_archivo)} líneas desde el archivo')

# También probar con un archivo inexistente
logs_inexistente = leer_logs_desde_archivo('/tmp/no_existe.log')
print(f'Archivo inexistente → {len(logs_inexistente)} líneas (manejo correcto del error)')

Leídas 20 líneas desde el archivo
Archivo inexistente → 0 líneas (manejo correcto del error)


## 6. Buenas prácticas — credenciales

**La regla de oro:** nunca hardcodear credenciales en el código. Esto aplica a API keys, passwords, tokens, todo.

Tres niveles de seguridad para manejar credenciales:

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/dos_and_donts.png" width="460"/>
<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/credenciales.png" width="380"/>

In [8]:
# MAL — nunca hacer esto
# SIEM_API_KEY = "abc123secreto"
# DB_PASSWORD = "password123"

# variables de entorno
SIEM_URL = os.environ.get('SIEM_URL', 'https://siem.empresa.internal')
SIEM_API_KEY = os.environ.get('SIEM_API_KEY', '')

if not SIEM_API_KEY:
    logger.warning('SIEM_API_KEY no configurada — algunas funciones no estarán disponibles')

print(f'SIEM URL configurada: {SIEM_URL}')
print(f'API Key presente: {bool(SIEM_API_KEY)}')

SIEM URL configurada: https://siem.empresa.internal
API Key presente: False


## 7. Integrando todo: análisis completo

Función que encapsula todo lo anterior y genera el reporte.

In [9]:
def analizar_sesion_soc(logs: list[str]) -> dict:
    """
    Análisis completo de una sesión de logs SSH.
    Devuelve un reporte listo para handoff de turno.
    """
    logger.info(f'Iniciando análisis de {len(logs)} líneas de log')

    # Parsear
    eventos = [parsear_log_ssh(l) for l in logs]
    eventos = [e for e in eventos if e is not None]

    # Analizar
    resumen = generar_resumen(eventos)
    sospechosas = detectar_fuerza_bruta(eventos, umbral=3)

    # Usuarios con acceso exitoso (para verificar si alguna IP sospechosa logró entrar)
    accesos_exitosos = [
        {'ip': e['ip'], 'usuario': e['usuario']}
        for e in eventos if e['tipo'] == 'accepted'
    ]

    # ¿Alguna IP sospechosa logró acceso exitoso? Señal de alarma alta
    ips_sospechosas = {s['ip'] for s in sospechosas}
    comprometidos = [a for a in accesos_exitosos if a['ip'] in ips_sospechosas]

    # Determinar recomendación
    if comprometidos:
        recomendacion = '🔴 CRITICO: IPs con fuerza bruta lograron acceso. Investigar inmediatamente.'
    elif sospechosas:
        recomendacion = f'🟡 ATENCIÓN: {len(sospechosas)} IP(s) con fuerza bruta activa. Considerar bloqueo en firewall.'
    else:
        recomendacion = '🟢 Sin anomalías detectadas en este período.'

    return {
        'periodo_analizado': f'{logs[0][:14]} → {logs[-1][:14]}',
        'total_eventos': resumen['total_eventos'],
        'intentos_fallidos': resumen['intentos_fallidos'],
        'accesos_exitosos': accesos_exitosos,
        'ips_fuerza_bruta': [s['ip'] for s in sospechosas],
        'posibles_compromisos': comprometidos,
        'recomendacion': recomendacion
    }


# Ejecutar análisis completo
reporte = analizar_sesion_soc(LOGS_MUESTRA)

print('=' * 55)
print('      REPORTE DE TURNO — SOC ANÁLISIS SSH')
print('=' * 55)
print(f"Período:           {reporte['periodo_analizado']}")
print(f"Total eventos:     {reporte['total_eventos']}")
print(f"Intentos fallidos: {reporte['intentos_fallidos']}")
print(f"Accesos exitosos:  {len(reporte['accesos_exitosos'])}")
print()
print(f"IPs fuerza bruta: {reporte['ips_fuerza_bruta']}")
print(f"Posibles compromisos: {reporte['posibles_compromisos']}")
print()
print(f"Recomendación: {reporte['recomendacion']}")
print('=' * 55)

# Guardar como JSON para el próximo turno
with open('reporte_turno.json', 'w') as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)
print('\nReporte guardado en reporte_turno.json')

      REPORTE DE TURNO — SOC ANÁLISIS SSH
Período:           Dec 10 06:55:4 → Dec 10 07:30:0
Total eventos:     16
Intentos fallidos: 10
Accesos exitosos:  2

IPs fuerza bruta: ['173.234.31.186', '45.33.32.156']
Posibles compromisos: []

Recomendación: 🟡 ATENCIÓN: 2 IP(s) con fuerza bruta activa. Considerar bloqueo en firewall.

Reporte guardado en reporte_turno.json


## 8. Modularidad — Organizar el código como si viviera en producción

Todo el código que escribiste hasta acá funciona. Pero si queda solo en el notebook, a los tres días nadie sabe dónde está cada función, se duplica lógica entre notebooks, y no podés reutilizarla desde un script o un test.

**La solución: módulos por responsabilidad.** Este notebook ya tiene los archivos físicos al lado:

```
02-python-para-analistas-soc/
├── 02_python_para_analistas_soc.ipynb  ← este archivo
└── soc/
    ├── __init__.py      ← from soc import parsear_log_ssh
    ├── parsers.py       ← PATRONES y parsear_log_ssh()
    ├── detectors.py     ← detectar_fuerza_bruta()
    └── reporters.py     ← generar_resumen(), analizar_sesion_soc()
```

Cada archivo tiene **una sola razón para existir**. Cuando cambia el formato del log (y va a cambiar), solo tocás `parsers.py`. Los tests apuntan al mismo código que usa producción — si el test pasa, el script en prod también pasa.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/monolitico.png" width="300"/>
<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/modularizado.png" width="300"/>

In [10]:
# Explorar la estructura del paquete soc/
import os
base = Path(MODULE_PATH) / 'soc'
print('Estructura del paquete soc/:')
for archivo in sorted(base.rglob('*.py')):
    lineas = len(archivo.read_text().splitlines())
    print(f'  {archivo.relative_to(MODULE_PATH)}  ({lineas} líneas)')

print()
# Leer el __init__.py para ver qué exporta la API pública
print('── soc/__init__.py (API pública) ──')
print((base / '__init__.py').read_text())

Estructura del paquete soc/:
  soc/__init__.py  (18 líneas)
  soc/detectors.py  (41 líneas)
  soc/parsers.py  (41 líneas)
  soc/reporters.py  (65 líneas)

── soc/__init__.py (API pública) ──
"""
soc — paquete de utilidades para análisis de logs SSH en el SOC.

API pública: importá directamente desde `soc` sin conocer la estructura interna.

    from soc import parsear_log_ssh, detectar_fuerza_bruta, generar_resumen
"""
from soc.parsers   import parsear_log_ssh, PATRONES
from soc.detectors import detectar_fuerza_bruta
from soc.reporters import generar_resumen, analizar_sesion_soc

__all__ = [
    'parsear_log_ssh',
    'PATRONES',
    'detectar_fuerza_bruta',
    'generar_resumen',
    'analizar_sesion_soc',
]



In [11]:
# Importar desde el paquete real — mismo código que usarías en producción
from soc import parsear_log_ssh as parsear_soc
from soc import detectar_fuerza_bruta as detectar_soc
from soc import generar_resumen as resumir_soc

eventos_soc = [e for e in (parsear_soc(l) for l in LOGS_MUESTRA) if e]
sospechosas_soc = detectar_soc(eventos_soc, umbral=3)
print(f'\nfrom soc import detectar_fuerza_bruta:')
print(f'  {len(sospechosas_soc)} IPs sospechosas — mismo resultado que arriba')

print()
print('La diferencia con el notebook:')
print('  Notebook  → define la función en la celda')
print('  soc/      → importa la función desde el archivo físico')
print('  Resultado → idéntico, pero el módulo es reutilizable desde cualquier script')


from soc import detectar_fuerza_bruta:
  2 IPs sospechosas — mismo resultado que arriba

La diferencia con el notebook:
  Notebook  → define la función en la celda
  soc/      → importa la función desde el archivo físico
  Resultado → idéntico, pero el módulo es reutilizable desde cualquier script


## 9. Resiliencia — Código que no se cae cuando el entorno falla

En producción, las cosas fallan: el SIEM no responde, la red tiene timeouts, el archivo de logs llega corrupto. Un script que explota con un traceback a las 3AM no es útil.

**Resiliencia significa dos cosas:**
1. **Reintentar** lo que puede funcionar en el segundo intento (APIs, red) — con backoff exponencial para no saturar el sistema que ya está fallando
2. **Degradar gracefully** lo que no tiene arreglo (logs corruptos, campos faltantes) — procesá lo que podés, registrá lo que no pudiste, nunca pierdas toda la sesión por una línea mala

La diferencia con "ignorar errores": en degradación graceful **siempre sabés qué falló** porque todo queda registrado en el log.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/resiliencia.png" width="460"/>

In [44]:
import time
import functools

# ─── Decorador de reintentos con backoff exponencial ──────────────────────────

def reintentar(max_intentos: int = 3, delay_base: float = 1.0,
               excepciones: tuple = (Exception,)):
    """
    Reintenta una función hasta max_intentos veces ante las excepciones indicadas.
    Espera: 1s → 2s → 4s (backoff exponencial). Loguea cada intento.
    """
    def decorador(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            ultimo_error = None
            for intento in range(1, max_intentos + 1):
                try:
                    return func(*args, **kwargs)
                except excepciones as e:
                    ultimo_error = e
                    if intento < max_intentos:
                        espera = delay_base * (2 ** (intento - 1))
                        logger.warning(
                            f'{func.__name__} falló (intento {intento}/{max_intentos}): {e}. '
                            f'Reintentando en {espera:.1f}s...'
                        )
                        time.sleep(espera)
                    else:
                        logger.error(
                            f'{func.__name__} falló definitivamente tras {max_intentos} intentos: {e}'
                        )
            raise ultimo_error
        return wrapper
    return decorador


# ─── Parseo seguro: nunca lanza excepciones, siempre loguea ──────────────────

def parsear_log_ssh_seguro(linea) -> dict | None:
    """
    Versión resiliente de parsear_log_ssh.
    Rechaza tipos incorrectos y líneas anómalas antes de procesar.
    Nunca interrumpe el pipeline con una excepción no controlada.
    """
    if not isinstance(linea, str):
        logger.warning(f'parsear_log_ssh: tipo inesperado {type(linea).__name__}, ignorando')
        return None
    if len(linea) > 2048:
        logger.warning(f'parsear_log_ssh: línea demasiado larga ({len(linea)} chars), ignorando')
        return None
    try:
        return parsear_log_ssh(linea)
    except Exception as e:
        logger.error(f'parsear_log_ssh: error inesperado: {e} | raw={linea[:80]}')
        return None


# ─── API externa con resiliencia ──────────────────────────────────────────────

@reintentar(max_intentos=3, delay_base=0.05, excepciones=(ConnectionError, TimeoutError))
def consultar_reputacion_ip(ip: str) -> dict:
    """
    Consulta una API de reputación de IPs (simulada).
    El decorador @reintentar la reintenta automáticamente si falla.
    En demo: falla con probabilidad 2/3 para mostrar el backoff.
    """
    import random
    if random.random() < 0.7:
        raise ConnectionError(f'Timeout consultando reputación de {ip}')
    return {'ip': ip, 'reputacion': 'maliciosa', 'puntaje': 85}


print('Funciones de resiliencia definidas:')
print('  @reintentar               → backoff exponencial configurable')
print('  parsear_log_ssh_seguro()  → parseo que nunca crashea el pipeline')
print('  consultar_reputacion_ip() → API externa con reintentos automáticos')

Funciones de resiliencia definidas:
  @reintentar               → backoff exponencial configurable
  parsear_log_ssh_seguro()  → parseo que nunca crashea el pipeline
  consultar_reputacion_ip() → API externa con reintentos automáticos


In [45]:
import random
random.seed(99)  # Seed fija para resultado reproducible en demo

# ─── Demo 1: parseo seguro con datos malformados ─────────────────────────────
print('─── Parseo seguro con entradas problemáticas ──────────────────────────')
entradas_problematicas = [
    None,                          # None en vez de string
    42,                            # Tipo incorrecto
    '',                            # Línea vacía
    'x' * 3000,                    # Línea gigante (log corrupto)
    'Dec 10 07:15:22 LabSZ sshd[100]: Failed password for root from 1.2.3.4 port 22 ssh2',  # OK
]

resultados_seguros = [parsear_log_ssh_seguro(e) for e in entradas_problematicas]
for entrada, resultado in zip(entradas_problematicas, resultados_seguros):
    tipo_entrada = type(entrada).__name__ if entrada is not None else 'None'
    etiqueta = f'OK → tipo={resultado["tipo"]}' if resultado else 'None (manejado)'
    print(f'  {tipo_entrada:<8} → {etiqueta}')

# ─── Demo 2: batch con líneas corruptas intercaladas ────────────────────────
print()
print('─── Procesamiento resiliente de batch ─────────────────────────────────')
logs_con_corrupcion = LOGS_MUESTRA[:5] + [None, 'CORRUPTO!!!', '', 'x' * 500] + LOGS_MUESTRA[5:10]

eventos_ok   = [r for linea in logs_con_corrupcion if (r := parsear_log_ssh_seguro(linea))]
rechazadas   = sum(1 for l in logs_con_corrupcion if l and not parsear_log_ssh_seguro(l))

print(f'  Total entradas:            {len(logs_con_corrupcion)}')
print(f'  Eventos parseados OK:      {len(eventos_ok)}')
print(f'  Sin crash (nulos/corruptos): True')
print()
print('El pipeline no se interrumpió. Los errores quedaron en el log.')

# ─── Demo 3: API con reintentos automáticos ──────────────────────────────────
print()
print('─── API externa: backoff exponencial ──────────────────────────────────')
random.seed(1)  # Este seed garantiza que falla los primeros intentos
try:
    resultado = consultar_reputacion_ip('173.234.31.186')
    print(f'  Éxito tras reintentos: {resultado}')
except ConnectionError as e:
    print(f'  Falló tras 3 intentos: {e}')
    print('  El sistema puede continuar con las demás IPs.')

─── Parseo seguro con entradas problemáticas ──────────────────────────
  None     → None (manejado)
  int      → None (manejado)
  str      → None (manejado)
  str      → None (manejado)
  str      → OK → tipo=failed_password

─── Procesamiento resiliente de batch ─────────────────────────────────
  Total entradas:            14
  Eventos parseados OK:      8
  Sin crash (nulos/corruptos): True

El pipeline no se interrumpió. Los errores quedaron en el log.

─── API externa: backoff exponencial ──────────────────────────────────
  Éxito tras reintentos: {'ip': '173.234.31.186', 'reputacion': 'maliciosa', 'puntaje': 85}


## 10. Testing con pytest — Verificá que tu detector funciona

**¿Por qué testear código de seguridad?**

Un bug en un parser de logs o en un detector de fuerza bruta tiene consecuencias directas: alertas que no se generan, IPs que no se bloquean, incidentes que no se detectan. La consecuencia de un bug en tu detector no es un mensaje de error — es un atacante que pasa desapercibido.

Pytest permite definir los casos que tu código *debe* manejar correctamente y ejecutarlos en segundos cada vez que cambiás algo.

Dos conceptos clave de pytest:
- **Fixture**: datos de prueba reutilizables entre tests (`@pytest.fixture`) — definís los datos de prueba una vez, los usás en todos los tests que los necesiten
- **Clase de tests**: agrupá tests relacionados para que el output de pytest sea fácil de leer y el scope quede claro

Los tests que vamos a escribir cubren los casos borde que más importan en producción: líneas malformadas, listas vacías, umbrales exactos, orden de resultados.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/test.png" width="460"/>

In [26]:
%%writefile test_soc.py
# test_soc.py — tests de unidad del paquete soc/
# Correr con: pytest test_soc.py -v
#
# Importa desde el paquete real (soc/), no desde soc_utils.
# Esto garantiza que los tests cubren exactamente el código que va a producción.
import pytest
from soc import parsear_log_ssh, detectar_fuerza_bruta, generar_resumen


# ─── Fixtures: datos de prueba reutilizables entre tests ──────────────────────

@pytest.fixture
def linea_failed_password():
    return 'Dec 10 07:15:22 LabSZ sshd[100]: Failed password for root from 173.234.31.186 port 39001 ssh2'

@pytest.fixture
def linea_invalid_user():
    return 'Dec 10 06:55:46 LabSZ sshd[200]: Invalid user webmaster from 173.234.31.186'

@pytest.fixture
def linea_accepted():
    return 'Dec 10 07:20:11 LabSZ sshd[220]: Accepted publickey for deploy from 10.0.0.15 port 60001 ssh2'

@pytest.fixture
def eventos_fuerza_bruta():
    """6 intentos desde la misma IP — debe activar detección con umbral=5."""
    logs = [
        f'Dec 10 07:15:{20+i} LabSZ sshd[100]: Failed password for root from 1.2.3.4 port {39000+i} ssh2'
        for i in range(6)
    ]
    return [e for e in (parsear_log_ssh(l) for l in logs) if e]


# ─── Tests de parseo ──────────────────────────────────────────────────────────

class TestParsearLogSSH:

    def test_parsea_failed_password(self, linea_failed_password):
        r = parsear_log_ssh(linea_failed_password)
        assert r is not None
        assert r['tipo'] == 'failed_password'
        assert r['ip'] == '173.234.31.186'
        assert r['usuario'] == 'root'

    def test_parsea_invalid_user(self, linea_invalid_user):
        r = parsear_log_ssh(linea_invalid_user)
        assert r is not None
        assert r['tipo'] == 'invalid_user'
        assert r['usuario'] == 'webmaster'
        assert r['ip'] == '173.234.31.186'

    def test_parsea_accepted_publickey(self, linea_accepted):
        r = parsear_log_ssh(linea_accepted)
        assert r is not None
        assert r['tipo'] == 'accepted'
        assert r['metodo'] == 'publickey'
        assert r['usuario'] == 'deploy'
        assert r['ip'] == '10.0.0.15'

    def test_retorna_none_en_linea_sin_match(self):
        linea = 'Dec 10 06:55:48 LabSZ sshd[200]: Connection closed by 173.234.31.186 [preauth]'
        assert parsear_log_ssh(linea) is None

    def test_retorna_none_en_linea_vacia(self):
        assert parsear_log_ssh('') is None

    def test_incluye_campo_raw(self, linea_failed_password):
        r = parsear_log_ssh(linea_failed_password)
        assert r['raw'] == linea_failed_password

    def test_failed_password_invalid_user_variant(self):
        linea = 'Dec 10 06:55:48 LabSZ sshd[200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2'
        r = parsear_log_ssh(linea)
        assert r is not None
        assert r['tipo'] == 'failed_password'
        assert r['usuario'] == 'webmaster'


# ─── Tests de detección de fuerza bruta ───────────────────────────────────────

class TestDetectarFuerzaBruta:

    def test_detecta_ip_con_muchos_intentos(self, eventos_fuerza_bruta):
        resultado = detectar_fuerza_bruta(eventos_fuerza_bruta, umbral=5)
        assert len(resultado) == 1
        assert resultado[0]['ip'] == '1.2.3.4'
        assert resultado[0]['intentos'] == 6

    def test_no_detecta_ip_bajo_umbral(self, eventos_fuerza_bruta):
        resultado = detectar_fuerza_bruta(eventos_fuerza_bruta[:3], umbral=5)
        assert resultado == []

    def test_riesgo_alto_cuando_supera_doble_umbral(self, eventos_fuerza_bruta):
        resultado = detectar_fuerza_bruta(eventos_fuerza_bruta * 2, umbral=5)
        assert resultado[0]['riesgo'] == 'ALTO'

    def test_riesgo_medio_cuando_supera_umbral_simple(self, eventos_fuerza_bruta):
        resultado = detectar_fuerza_bruta(eventos_fuerza_bruta, umbral=5)
        assert resultado[0]['riesgo'] == 'MEDIO'

    def test_lista_vacia_retorna_lista_vacia(self):
        assert detectar_fuerza_bruta([]) == []

    def test_ordena_por_intentos_descendente(self):
        eventos_a = [{'tipo': 'failed_password', 'ip': 'IP_A', 'usuario': 'u'}] * 3
        eventos_b = [{'tipo': 'failed_password', 'ip': 'IP_B', 'usuario': 'u'}] * 8
        resultado = detectar_fuerza_bruta(eventos_a + eventos_b, umbral=3)
        assert resultado[0]['ip'] == 'IP_B'

    def test_recolecta_todos_los_usuarios_probados(self, eventos_fuerza_bruta):
        resultado = detectar_fuerza_bruta(eventos_fuerza_bruta, umbral=5)
        assert 'root' in resultado[0]['usuarios_probados']

    def test_ignora_eventos_de_tipo_accepted(self):
        eventos = [{'tipo': 'accepted', 'ip': '1.2.3.4', 'usuario': 'admin'}] * 10
        assert detectar_fuerza_bruta(eventos, umbral=5) == []


# ─── Tests del generador de resumen ───────────────────────────────────────────

class TestGenerarResumen:

    def test_cuenta_intentos_fallidos_y_exitosos(self):
        eventos = (
            [{'tipo': 'failed_password', 'ip': '1.1.1.1', 'usuario': 'root'}] * 5 +
            [{'tipo': 'accepted',        'ip': '192.168.1.1', 'usuario': 'admin'}] * 2
        )
        r = generar_resumen(eventos)
        assert r['intentos_fallidos'] == 5
        assert r['accesos_exitosos'] == 2
        assert r['total_eventos'] == 7

    def test_con_lista_vacia(self):
        r = generar_resumen([])
        assert r['total_eventos'] == 0
        assert r['intentos_fallidos'] == 0
        assert r['accesos_exitosos'] == 0
        assert r['ips_fuerza_bruta'] == 0

    def test_detecta_fuerza_bruta_integrado(self):
        eventos = [{'tipo': 'failed_password', 'ip': '6.6.6.6', 'usuario': 'root'}] * 6
        r = generar_resumen(eventos)
        assert r['ips_fuerza_bruta'] == 1
        assert r['detalle_sospechosas'][0]['ip'] == '6.6.6.6'

Writing test_soc.py


In [ ]:
!PYTHONPATH={MODULE_PATH} python3 -m pytest test_soc.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.7.30, typeguard-4.5.1
collected 18 items                                                             

test_soc.py ..................                                           [100%]

============================== 18 passed in 0.02s ==============================
